<a href="https://colab.research.google.com/github/itsmareena/titanic-survival-prediction/blob/main/titanic_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
from google.colab import files
uploaded = files.upload()

Saving gender_submission.csv to gender_submission (1).csv
Saving test.csv to test (1).csv
Saving train.csv to train (1).csv


In [44]:
import pandas as pd

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [25]:
print(train.shape)
train.info()

(891, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [26]:
train['Survived'].value_counts()

,count
Survived,
0,549
1,342


In [27]:
train = train.drop('Cabin', axis=1)

In [28]:
print(train['Embarked'].mode()[0])   # see the most common value
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])

S


In [29]:
train['Age'] = train['Age'].fillna(train['Age'].median())

In [30]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          891 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Embarked     891 non-null    object 
dtypes: float64(2), int64(5), object(4)
memory usage: 76.7+ KB


In [31]:
train = train.drop(['PassengerId', 'Name', 'Ticket'], axis=1)

In [32]:
train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})

In [33]:
train = pd.get_dummies(train, columns=['Embarked'], drop_first=True)

In [34]:
train.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
0,0,3,0,22.0,1,0,7.2500,False,True
1,1,1,1,38.0,1,0,71.2833,False,False
2,1,3,1,26.0,0,0,7.9250,False,True
3,1,1,1,35.0,1,0,53.1000,False,True
4,0,3,0,35.0,0,0,8.0500,False,True


In [35]:
train['Embarked_Q'] = train['Embarked_Q'].astype(int)
train['Embarked_S'] = train['Embarked_S'].astype(int)
train.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
0,0,3,0,22.0,1,0,7.2500,0,1
1,1,1,1,38.0,1,0,71.2833,0,0
2,1,3,1,26.0,0,0,7.9250,0,1
3,1,1,1,35.0,1,0,53.1000,0,1
4,0,3,0,35.0,0,0,8.0500,0,1


In [36]:
from sklearn.model_selection import train_test_split

X = train.drop('Survived', axis=1)   # all columns except the answer
y = train['Survived']                 # the answer column

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_val.shape)

(712, 8) (179, 8)


In [37]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

predictions = model.predict(X_val)
accuracy = accuracy_score(y_val, predictions)
print("Accuracy:", accuracy)

Accuracy: 0.8100558659217877


In [38]:
import pandas as pd
coefficients = pd.DataFrame({'feature': X_train.columns, 'coefficient': model.coef_[0]})
print(coefficients.sort_values('coefficient', ascending=False))

      feature  coefficient
1         Sex     2.591222
5        Fare     0.002576
2         Age    -0.030574
4       Parch    -0.107848
6  Embarked_Q    -0.111981
3       SibSp    -0.295062
7  Embarked_S    -0.400561
0      Pclass    -0.938047


In [39]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_val, predictions))

[[90 15]
 [19 55]]


In [40]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_val)
print("Random Forest Accuracy:", accuracy_score(y_val, rf_predictions))
print(confusion_matrix(y_val, rf_predictions))

Random Forest Accuracy: 0.7988826815642458
[[88 17]
 [19 55]]


In [41]:
import pandas as pd

# Get the feature names and their coefficients
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)

print(feature_importance)

      feature  coefficient
1         Sex     2.591222
5        Fare     0.002576
2         Age    -0.030574
4       Parch    -0.107848
6  Embarked_Q    -0.111981
3       SibSp    -0.295062
7  Embarked_S    -0.400561
0      Pclass    -0.938047


In [48]:
# Reload test fresh
test = pd.read_csv('test.csv')
test_passenger_ids = test['PassengerId'].copy()

# Clean test using train's stats
test = test.drop('Cabin', axis=1)
test['Age'] = test['Age'].fillna(train['Age'].median())
test['Embarked'] = test['Embarked'].fillna(train['Embarked'].mode()[0])
test['Sex'] = test['Sex'].map({'male': 0, 'female': 1})
test = pd.get_dummies(test, columns=['Embarked'], drop_first=True)
test = test.drop(['PassengerId', 'Name', 'Ticket'], axis=1)

# Check which columns have NaN
print(test.isnull().sum())

# Fill any remaining NaN with the median from training data
for col in test.columns:
    if test[col].isnull().sum() > 0:
        test[col] = test[col].fillna(train[col].median())

print("\nAfter filling:")
print(test.isnull().sum())

# Predict with your trained model (whichever one you have: LogisticRegression or RandomForest)
test_predictions = model_final.predict(test)

# Create submission
submission = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Survived': test_predictions
})



Pclass        0
Sex           0
Age           0
SibSp         0
Parch         0
Fare          1
Embarked_Q    0
Embarked_S    0
dtype: int64

After filling:
Pclass        0
Sex           0
Age           0
SibSp         0
Parch         0
Fare          0
Embarked_Q    0
Embarked_S    0
dtype: int64


In [49]:
submission.to_csv('submission.csv', index=False)
print(submission.head())
print("Submission created!")

   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
Submission created!
